# 02 – Report extraction experiments

History of what was tried in `workers/extraction` and what it did to the scores.

Every `python -m workers.extraction experiment <name> --note "<hypothesis>"` evaluates one run
against the hand-checked ground truth (`data/ground_truth/`, git-ignored) and saves a record in
`workers/extraction/experiments/runs/`. A record holds the configuration (model, prompt and schema
fingerprints, git commit, gold-set hash) and the scores – never report text – so this notebook
reads only those records. The reasoning behind each experiment is in
`workers/extraction/experiments/EXPERIMENTS.md`.

**Caveat:** the gold set has 12 reports. One document moves a field's accuracy by ~8 pp, so treat
differences of a few points as noise until confidence intervals are added.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "workers" / "extraction").is_dir())
RUNS = ROOT / "workers" / "extraction" / "experiments" / "runs"

records = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(RUNS.glob("[0-9][0-9][0-9]_*.json"))]
print(f"{len(records)} experiments in {RUNS.relative_to(ROOT)}")


def pct(frame, digits=0):
    """Percent strings for display (no Styler, so no jinja2 needed)."""
    return frame.map(lambda v: "–" if pd.isna(v) else f"{v:.{digits}%}")

## 1. Experiments and what changed

`changed` lists the configuration keys that differ from the previous experiment – the variable
that experiment tested. `gold` flags runs scored against a different version of the ground truth
(not directly comparable).

In [ ]:
def changed(prev, cur):
    if prev is None:
        return "(first)"
    keys = [k for k, v in cur["config"].items() if prev["config"].get(k) != v]
    if prev["input"] != cur["input"]:
        keys.append("input")
    if prev["code"]["git_commit"] != cur["code"]["git_commit"]:
        keys.append("code")
    return ", ".join(keys) or "–"

latest_gold = records[-1]["data"]["gold_sha"] if records else None
rows = []
for prev, cur in zip([None, *records], records):
    rows.append({
        "id": cur["id"],
        "name": cur["name"],
        "date": cur["created_at"][:10],
        "model": cur["config"]["llm_model"],
        "prompt": cur["config"]["prompt_version"],
        "schema": cur["config"]["schema_version"],
        "few_shot": cur["config"]["few_shot"],
        "structured": cur["config"]["structured_output"],
        "input": cur["input"],
        "changed": changed(prev, cur),
        "gold": "" if cur["data"]["gold_sha"] == latest_gold else "other",
        "failed": cur["runtime"]["failed"],
        "llm_s": cur["runtime"]["llm_seconds_mean"],
        "hypothesis": cur["hypothesis"],
    })
experiments = pd.DataFrame(rows).set_index("id")
experiments

## 2. Headline metrics over time

Pooled field accuracy, entity F1 (were lesions / nodes found at all) and BI-RADS of the
conclusion. A good change moves these up without pushing others down.

In [ ]:
summary = pd.DataFrame(
    {r["id"]: r["summary"] for r in records}
).T.astype(float)
summary.index.name = "id"
display(pct(summary))

headline = ["field_accuracy", "zmiany_f1", "wezly_f1", "obserwacje_f1", "wniosek_birads_accuracy"]
fig, ax = plt.subplots(figsize=(10, 4))
for metric in headline:
    if summary[metric].notna().any():
        ax.plot(summary.index, summary[metric], marker="o", label=metric)
ax.set_ylim(0, 1.05)
ax.set_xlabel("experiment")
ax.set_ylabel("score")
ax.set_title("Headline metrics per experiment")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.show()

## 3. Accuracy per field group

Where the errors are: breasts, MMG / USG observations, lesions, nodes, conclusion.

In [ ]:
groups = [c for c in summary.columns if c.endswith("_accuracy") and c not in ("field_accuracy", "wniosek_birads_accuracy")]
ax = summary[groups].plot(kind="bar", figsize=(11, 4), ylim=(0, 1.05), rot=0)
ax.set_title("Accuracy per field group")
ax.legend(loc="lower right", fontsize=8)
ax.grid(axis="y", alpha=0.3)
plt.show()

## 4. Field × experiment heatmap

Accuracy of every scored field (rows) in every experiment (columns). Fields that improved or got
worse between neighbouring experiments are what the change actually affected.

In [ ]:
fields = pd.DataFrame({
    r["id"]: {name: s["accuracy"] for name, s in r["results"]["fields"].items()}
    for r in records
}).sort_index().astype(float)

fig, ax = plt.subplots(figsize=(1.2 * len(fields.columns) + 4, 0.28 * len(fields) + 1))
im = ax.imshow(fields.values, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks(range(len(fields.columns)), fields.columns)
ax.set_yticks(range(len(fields.index)), fields.index, fontsize=8)
for (i, j), value in pd.DataFrame(fields.values).stack().items():
    ax.text(j, i, f"{value:.0%}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.02)
ax.set_title("Field accuracy per experiment")
plt.show()

### Biggest changes against the previous experiment

In [ ]:
if len(fields.columns) >= 2:
    delta = fields.diff(axis=1).iloc[:, 1:]
    moves = delta.stack().rename("delta").reset_index()
    moves.columns = ["field", "experiment", "delta"]
    moves = moves[moves["delta"].abs() >= 0.05].sort_values("delta")
    moves["delta"] = moves["delta"].map(lambda v: f"{v:+.0%}")
    display(moves)
else:
    print("Needs at least two experiments.")

## 5. What OCR costs

Pairs of experiments with the same model, prompt and schema – one on hand-typed transcripts
(LLM alone), one on the scans (OCR + LLM). The gap is the price of OCR errors; the scans run also
reports the OCR character error rate.

In [ ]:
llm_keys = ["llm_model", "prompt_sha", "schema_sha", "few_shot", "structured_output"]
by_config = {}
for r in records:
    by_config.setdefault(tuple(r["config"][k] for k in llm_keys), {})[r["input"]] = r

pairs = []
for config, runs in by_config.items():
    if {"transcripts", "scans"} <= runs.keys():
        t, s = runs["transcripts"], runs["scans"]
        for metric in headline:
            pairs.append({
                "transcripts": t["id"], "scans": s["id"], "metric": metric,
                "llm alone": t["summary"][metric], "ocr + llm": s["summary"][metric],
            })
        pairs.append({"transcripts": t["id"], "scans": s["id"], "metric": "ocr_cer",
                      "llm alone": None, "ocr + llm": s["summary"]["ocr_cer"]})
if pairs:
    table = pd.DataFrame(pairs)
    table[["llm alone", "ocr + llm"]] = pct(table[["llm alone", "ocr + llm"]].astype(float), 1)
    display(table)
else:
    print("No transcript/scan pair with the same configuration yet.")